# Opioid Use Disorder (OUD) Risk Prediction
## Exploratory notebook sandbox

This notebook is an exploratory sandbox for Machine Learning Operations (MLOps) development

Use it to
- Inspect raw and cleaned data
- Perform focused Exploratory Data Analysis (EDA) checks to validate assumptions
- Run quick training and inference checks while iterating on features
- Experiment with new features and models to improve metrics performance

Do not use it to
- Replace the reproducible pipeline in `src/main.py`
- Store production logic in notebook cells
- Write production artifacts to disk from the notebook

Canonical production entry point
- Run from terminal: `python -m src.main`

Why this separation matters
- The orchestrator (`src/main.py`) is the factory: deterministic inputs, deterministic outputs, clear provenance
- The notebook is the lab bench: interactive inspection, rapid iteration, visible intermediate states and exploration

### A) Environment setup and imports

Learning intent
- Make imports reliable regardless of how you opened Jupyter
- Ensure relative paths behave the same way for everyone
- Keep notebook code thin by calling functions from `src/` modules

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import os # For changing the working directory to the repo root
import sys # For manipulating sys.path to ensure imports work regardless of where the notebook is started
from pathlib import Path # For working with file paths in a platform-independent way
import yaml # For loading configuration from config.yaml

import pandas as pd
from IPython.display import display # For displaying dataframes nicely in Jupyter

# Repo root alignment
# Problem this solves
# - Notebooks can start with different working directories depending on IDE and settings
# - Relative paths then break unpredictably
#
# Approach
# - Search upward from the current folder until we find a folder that contains `src/`
# - Set that as the working directory so relative paths match the orchestrator behaviour
def find_repo_root(start: Path, marker_dir: str = "src", max_hops: int = 12) -> Path:
    current = start.resolve()
    for _ in range(max_hops):
        if (current / marker_dir).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError(
        f"Could not find repo root containing '{marker_dir}/' starting from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
os.chdir(PROJECT_ROOT)

# Ensure `import src...` works even if Jupyter started elsewhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())

# Imports from production modules
# Note: we intentionally do NOT import from src/main.py
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference
from src.main import three_way_split

PROJECT_ROOT: /home/idiazl/2026_MLOps/3. mlops_CICD-repo
cwd: /home/idiazl/2026_MLOps/3. mlops_CICD-repo


### B) Sandbox configuration

Guideline
- Keep configuration in one place
- Use the same column names and defaults as the orchestrator where possible
- Prefer explicit lists over clever inference so errors are actionable

In [2]:
# Sandbox configuration (config-driven)
# Keeps notebook behavior aligned with src/main.py by reading config.yaml

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"config.yaml not found at {CONFIG_PATH}\n"
        "Open the correct repo root and ensure config.yaml exists"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

if not isinstance(cfg, dict):
    raise ValueError("config.yaml must parse into a dictionary")

# Short references for readability
paths = cfg["paths"]
problem = cfg["problem"]
split = cfg["split"]
features = cfg.get("features", {})
validation = cfg.get("validation", {})
run_cfg = cfg.get("run", {})
training = cfg.get("training", {})

# Resolve repo-relative paths (same pattern as src/main.py)
RAW_DATA_PATH = PROJECT_ROOT / paths["raw_data"]
CLEAN_DATA_PATH = PROJECT_ROOT / paths["processed_data"]
MODEL_PATH = PROJECT_ROOT / paths["model_artifact"]
INFERENCE_DATA_PATH = PROJECT_ROOT / paths["inference_data"]
PREDICTIONS_PATH = PROJECT_ROOT / paths["predictions_artifact"]

# Notebook-only teaching toggle
# Keep separate so students learn: config drives the system, notebook toggles are local experiments
EVALUATE_ON_TEST = False

print("[notebook] Loaded config.yaml")
print("[notebook] RAW_DATA_PATH:", RAW_DATA_PATH)

[notebook] Loaded config.yaml
[notebook] RAW_DATA_PATH: /home/idiazl/2026_MLOps/3. mlops_CICD-repo/data/raw/opiod_raw_data.csv


### 1) Load raw data (`src.load_data`)

Educational note
- We load raw data exactly once
- Raw data should be treated as immutable input

In [3]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found at {RAW_DATA_PATH}\n"
        "Fix the file location by updating config.yaml under paths.raw_data"
    )

df_raw = load_raw_data(RAW_DATA_PATH)
print("df_raw.shape:", df_raw.shape)
df_raw.head()

[load_data.load_raw_data] Loading raw data from /home/idiazl/2026_MLOps/3. mlops_CICD-repo/data/raw/opiod_raw_data.csv
[utils.load_csv] Loading CSV from /home/idiazl/2026_MLOps/3. mlops_CICD-repo/data/raw/opiod_raw_data.csv
[load_data.load_raw_data] Loaded dataframe shape: (1000, 22)
df_raw.shape: (1000, 22)


,ID,OD,Low_inc,SURG,rx ds,A,B,C,D,E,...,I,J,K,L,M,N,R,S,T,V
0,1,1,1,0,330,0,0,0,1,1,...,1,1,0,1,1,1,1,0,0,0
1,2,1,1,0,457,0,0,1,0,1,...,1,1,0,0,1,1,1,1,0,0
2,3,1,1,1,722,0,1,0,1,1,...,1,1,1,0,1,0,1,0,1,0
3,4,0,1,0,262,0,1,0,0,1,...,1,0,1,1,1,1,1,1,1,0
4,5,1,1,0,780,0,0,0,0,1,...,1,0,0,0,1,0,1,0,0,0


### 2) Focused EDA checks

Goal
- Validate assumptions before investing in feature engineering and training

Guideline
- Prefer small, targeted checks over long notebooks
- If an assumption is wrong, fix the pipeline, not the notebook

In [4]:
print("Missing values (top 10 columns):")
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

target_col = problem["target_column"]
if target_col in df_raw.columns:
    vc = df_raw[target_col].value_counts(dropna=False)
    print("\nTarget distribution:")
    display(vc)
    if vc.sum() > 0:
        print("Positive class prevalence:", float(vc.get(1, 0) / vc.sum()))
else:
    print(f"\nTarget column '{target_col}' not found in raw data")

# Teaching variable: raw dataset may contain 'rx ds' or 'rx_ds'
rx_col = "rx_ds" if "rx_ds" in df_raw.columns else ("rx ds" if "rx ds" in df_raw.columns else None)
if rx_col is not None:
    print(f"\nSummary for '{rx_col}':")
    display(df_raw[rx_col].describe())

print("\nBinary indicator check (sample of 3 columns):")
for col in list(features.get("binary_sum_cols", []))[:3]:
    print(f"'{col}' unique values:", pd.Series(df_raw[col].unique()).head(10).tolist())

Missing values (top 10 columns):


ID         0
OD         0
Low_inc    0
SURG       0
rx ds      0
A          0
B          0
C          0
D          0
E          0
dtype: int64


Target distribution:


OD
0    819
1    181
Name: count, dtype: int64

Positive class prevalence: 0.181

Summary for 'rx ds':


count    1000.000000
mean      329.961000
std       333.335543
min         1.000000
25%        60.000000
50%       182.000000
75%       589.250000
max      1699.000000
Name: rx ds, dtype: float64


Binary indicator check (sample of 3 columns):
'A' unique values: [0, 1]
'B' unique values: [0, 1]
'C' unique values: [0, 1]


### 3) Clean data (`src.clean_data`)

Educational note
- Cleaning should be deterministic
- Cleaning should not learn from the data in a way that leaks information across splits

In [5]:
df_clean = clean_dataframe(df_raw, target_column=problem["target_column"])
print("df_clean.shape:", df_clean.shape)
df_clean.head()

[clean_data.clean_dataframe] Cleaning dataframe
[clean_data.clean_dataframe] Rows after cleaning: 1000
df_clean.shape: (1000, 22)


,ID,OD,Low_inc,SURG,rx_ds,A,B,C,D,E,...,I,J,K,L,M,N,R,S,T,V
0,1,1,1,0,330,0,0,0,1,1,...,1,1,0,1,1,1,1,0,0,0
1,2,1,1,0,457,0,0,1,0,1,...,1,1,0,0,1,1,1,1,0,0
2,3,1,1,1,722,0,1,0,1,1,...,1,1,1,0,1,0,1,0,1,0
3,4,0,1,0,262,0,1,0,0,1,...,1,0,1,1,1,1,1,1,1,0
4,5,1,1,0,780,0,0,0,0,1,...,1,0,0,0,1,0,1,0,0,0


### 4) Didactic check: what changed after cleaning
- Make transformations visible
- Help you debug unexpected column name changes or dropped columns

In [6]:
raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed:")
display(removed_cols)

print("Columns added:")
display(added_cols)

print('Cleaned has "rx_ds":', "rx_ds" in df_clean.columns)

Columns removed:


['rx ds']

Columns added:


['rx_ds']

Cleaned has "rx_ds": True


### 5) Validate data (security gate)

Educational note
- Validation is a fail-fast gate
- It prevents wasting time on training with broken assumptions

Guideline
- If validation fails, fix the upstream module or the data contract
- Do not patch around failures inside the notebook

In [7]:
required_columns = (
    [problem["target_column"]]
    + list(features.get("quantile_bin", []))
    + list(features.get("categorical_onehot", []))
    + list(features.get("numeric_passthrough", []))
    + list(features.get("binary_sum_cols", []))
)
required_columns = list(dict.fromkeys(required_columns))

validate_dataframe(
    df=df_clean,
    required_columns=required_columns,
    check_missing_values=bool(validation.get("check_missing_values", False)),
    target_column=problem["target_column"],
    target_allowed_values=[0, 1] if problem["problem_type"] == "classification" else None,
    numeric_non_negative_cols=list(validation.get("numeric_non_negative_cols", [])),
)

print("[notebook] Validation passed")

[validate.validate_dataframe] Validating dataframe
[notebook] Validation passed


### 6) Build the feature recipe (`src.features`)

Educational note
- This step builds a preprocessing blueprint
- It must not fit on the full dataset in the notebook
- The recipe learns only when fitted on the training split inside the training pipeline

In [8]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=list(features.get("quantile_bin", [])),
    categorical_onehot_cols=list(features.get("categorical_onehot", [])),
    numeric_passthrough_cols=list(features.get("numeric_passthrough", [])),
    binary_sum_cols=list(features.get("binary_sum_cols", [])),
    n_bins=int(features.get("n_bins", 4)),
)

print("[notebook] Feature recipe built, not fitted yet")
preprocessor

[features.get_feature_preprocessor] Building feature recipe from configuration
[notebook] Feature recipe built, not fitted yet


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('quantile_bins', ...), ('binary_sum', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

### 7) Leakage gate: three-way split (train, validation, test)

Educational note
- Train split is the only split allowed to learn preprocessing parameters and model weights
- Validation split is for iteration and model selection
- Test split is a final audit vault

In [9]:
X = df_clean.drop(columns=[problem["target_column"]])
y = df_clean[problem["target_column"]]

if problem["problem_type"] == "classification" and y.nunique() < 2:
    raise ValueError(
        f"Target '{problem['target_column']}' has fewer than 2 classes after cleaning\n"
        "Classification training and stratified splitting cannot proceed"
    )

X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X,
    y,
    test_size=float(split["test_size"]),
    val_size=float(split["val_size"]),
    random_state=int(split["random_state"]),
    stratify=(problem["problem_type"] == "classification"),
)

print("[notebook] Split sizes")
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

[notebook] Split sizes
Train: (800, 21) Validation: (150, 21) Test: (50, 21)


### 8) Train (`src.train`)

Educational note
- This is the only step where `.fit()` happens
- The preprocessor and estimator are fitted only on training data

In [10]:
pt = problem["problem_type"].strip().lower()
model_params = dict(training.get(pt, {}) or {})

model_pipeline = train_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type=pt,
    model_params=model_params,
)

print("[notebook] Training complete")
model_pipeline

[train] Training model pipeline for problem_type=classification
[notebook] Training complete


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('quantile_bins', ...), ('binary_sum', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

### 9) Evaluate (`src.evaluate`)

**Educational Note: The Audit Vault**
* **Validation:** Used to iteratively guide our model and feature engineering decisions.
* **Test:** Our blind audit vault. We use a strict boolean flag (`evaluate_on_test` in our `SETTINGS`) to ensure we only peek at this vault when we are absolutely ready to finalize the model. Notice that our production orchestrator (`src/main.py`) doesn't even contain code to score the test set!

**Metric Note**
* `evaluate_model` automatically selects the appropriate metrics based on the `problem_type` and returns them as a dictionary.
* **Classification:** Returns **PR AUC** (Average Precision - our primary metric for imbalanced data) and **ROC AUC**.
* **Regression:** Returns **RMSE** (Root Mean Squared Error).

In [11]:
pt = problem["problem_type"].strip().lower()

val_metrics = evaluate_model(
    model=model_pipeline,
    X_eval=X_val,
    y_eval=y_val,
    problem_type=pt,
)
print(f"[notebook] Validation metrics: {val_metrics}")

# The "Break Glass" Audit Step
if EVALUATE_ON_TEST:
    test_metrics = evaluate_model(
        model=model_pipeline,
        X_eval=X_test,
        y_eval=y_test,
        problem_type=pt,
    )
    print(f"[notebook] Test metrics (audit vault): {test_metrics}")

[evaluate.evaluate_model] Starting evaluation
[evaluate.evaluate_model] Metrics={'pr_auc': 0.2525589727467549, 'roc_auc': 0.5965070761818728}
[notebook] Validation metrics: {'pr_auc': 0.2525589727467549, 'roc_auc': 0.5965070761818728}


### 10) Inference demo (`src.infer`)

Educational note
- Inference simulates what happens after training
- We deliberately use rows from the test split to simulate unseen cases

In [12]:
sample_n = min(10, len(X_test))
X_infer_sample = X_test.sample(n=sample_n, random_state=int(split["random_state"]))

pt = problem["problem_type"].strip().lower()
include_proba = (pt == "classification") and bool(run_cfg.get("include_proba_if_classification", True))

df_predictions = run_inference(
    model=model_pipeline,
    X_infer=X_infer_sample,
    include_proba=include_proba,
)

print("[notebook] Inference results")
display(df_predictions.head(10))

[infer.run_inference] Running inference
[notebook] Inference results


,prediction,proba
463,1,0.529023
397,1,0.608544
392,1,0.609097
585,1,0.528734
7,0,0.372084
267,0,0.448718
968,1,0.529745
101,0,0.449579
141,1,0.530034
943,1,0.529890


### 11) Inspect production artifacts produced by the orchestrator

This notebook does not write artifacts to disk

Use this cell only after you run the orchestrator from terminal
- `python -m src.main`

Goal
- Prove that the factory output exists on disk
- Inspect outputs without modifying them

In [13]:
from src.utils import load_model

try:
    clean_from_disk = pd.read_csv(CLEAN_DATA_PATH)
    preds_from_disk = pd.read_csv(PREDICTIONS_PATH)
    model_from_disk = load_model(MODEL_PATH)

    print("clean.csv shape:", clean_from_disk.shape)
    print("predictions.csv shape:", preds_from_disk.shape)
    print("loaded model type:", type(model_from_disk))

except FileNotFoundError as e:
    print("[notebook] Missing expected artifact")
    print(e)

[utils.load_model] Loading model from /home/idiazl/2026_MLOps/3. mlops_CICD-repo/models/model.joblib
clean.csv shape: (1000, 22)
predictions.csv shape: (10, 3)
loaded model type: <class 'sklearn.pipeline.Pipeline'>
